In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score


JSON_DIR = "../json"
LABEL_DIR = "../json_label"

MODEL_PATH = "../model/random_forest_special_assessment.pkl"


LEVEL_MAPPING = {
    "Không có": 0,
    "Chưa đủ cơ sở": 1,
    "Đang hình thành": 2,
    "Đạt": 3,
    "Nổi trội": 4
}

REVERSE_LEVEL_MAPPING = {
    0: "Không có",
    1: "Chưa đủ cơ sở",
    2: "Đang hình thành",
    3: "Đạt",
    4: "Nổi trội"
}


SUBJECTS = [
    "vietnamese",
    "mathematics",
    "informatics",
    "english",
    "music",
    "arts",
    "civics",
    "physical_education",
    "experiential_activities"
]


LABEL_FIELDS = [
    "Năng lực ngôn ngữ",
    "Năng lực tính toán",
    "Năng lực khoa học",
    "Năng lực công nghệ",
    "Năng lực tin học",
    "Năng lực thẩm mĩ",
    "Năng lực thể chất"
]

In [3]:
def load_input(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    features = []
    for subject in SUBJECTS:
        subject_data = data.get(subject, {})
        confident = subject_data.get("confident", 0)
        level = subject_data.get("level", 0)
        features.append(confident)
        features.append(level)
    return features

In [4]:
load_input("../json/20252026.01.Mo1.001.json")

[0.8557479977607727,
 1,
 0.6805649995803833,
 1,
 0.6744087934494019,
 1,
 0.5206648707389832,
 1,
 0.6389163732528687,
 1,
 0.7888104915618896,
 1,
 0.6668856143951416,
 1,
 0.6903708577156067,
 1,
 0.7037678360939026,
 2]

In [5]:
def load_label(label_path):
    with open(label_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    label_dict = {
        item["field"]: LEVEL_MAPPING[item["level"]]
        for item in data["labels"]
    }
    labels = []
    for field in LABEL_FIELDS:
        labels.append(label_dict.get(field, 0))
    return labels

In [6]:
load_label("../json_label/20252026.01.Mo1.001_label.json")

[2, 2, 2, 1, 2, 2, 2]

In [7]:
def build_dataset():
    X = []
    y = []
    student_codes = []

    for filename in os.listdir(JSON_DIR):
        if not filename.endswith(".json"):
            continue
        student_code = filename.replace(".json", "")
        input_path = os.path.join(JSON_DIR, filename)
        label_path = os.path.join(LABEL_DIR, f"{student_code}_label.json")
        if not os.path.exists(label_path):
            print(f"Bỏ qua {student_code}: không tìm thấy label")
            continue
        features = load_input(input_path)
        labels = load_label(label_path)
        X.append(features)
        y.append(labels)
        student_codes.append(student_code)
    return np.array(X), np.array(y), student_codes
build_dataset()

Bỏ qua 20252026.03.Ba1.001: không tìm thấy label
Bỏ qua 20252026.03.Ba1.002: không tìm thấy label
Bỏ qua 20252026.03.Ba1.004: không tìm thấy label
Bỏ qua 20252026.03.Ba1.005: không tìm thấy label
Bỏ qua 20252026.03.Ba1.006: không tìm thấy label
Bỏ qua 20252026.03.Ba1.007: không tìm thấy label
Bỏ qua 20252026.03.Ba1.008: không tìm thấy label
Bỏ qua 20252026.03.Ba1.009: không tìm thấy label
Bỏ qua 20252026.03.Ba1.010: không tìm thấy label
Bỏ qua 20252026.03.Ba1.011: không tìm thấy label
Bỏ qua 20252026.03.Ba1.012: không tìm thấy label
Bỏ qua 20252026.03.Ba1.013: không tìm thấy label
Bỏ qua 20252026.03.Ba1.014: không tìm thấy label
Bỏ qua 20252026.03.Ba1.015: không tìm thấy label
Bỏ qua 20252026.03.Ba1.016: không tìm thấy label
Bỏ qua 20252026.03.Ba1.017: không tìm thấy label
Bỏ qua 20252026.03.Ba1.018: không tìm thấy label
Bỏ qua 20252026.03.Ba1.019: không tìm thấy label
Bỏ qua 20252026.03.Ba1.020: không tìm thấy label
Bỏ qua 20252026.03.Ba1.021: không tìm thấy label
Bỏ qua 20252026.03.B

(array([[0.855748  , 1.        , 0.680565  , ..., 1.        , 0.70376784,
         2.        ],
        [0.84934038, 1.        , 0.77300429, ..., 1.        , 0.70376784,
         2.        ],
        [0.855748  , 1.        , 0.77300429, ..., 3.        , 0.70376784,
         2.        ],
        ...,
        [0.7630868 , 2.        , 0.62483573, ..., 1.        , 0.56927067,
         2.        ],
        [0.89787829, 2.        , 0.64339387, ..., 1.        , 0.84531188,
         2.        ],
        [0.79388118, 3.        , 0.69008452, ..., 1.        , 0.59735811,
         2.        ]], shape=(63, 18)),
 array([[2, 2, 2, 1, 2, 2, 2],
        [2, 2, 2, 1, 4, 2, 2],
        [2, 2, 2, 1, 2, 2, 4],
        [2, 2, 2, 1, 2, 2, 2],
        [2, 2, 2, 1, 2, 2, 2],
        [2, 2, 2, 1, 3, 2, 2],
        [2, 2, 2, 1, 2, 2, 4],
        [2, 2, 2, 1, 3, 2, 2],
        [2, 2, 2, 1, 2, 2, 2],
        [0, 0, 2, 1, 2, 2, 2],
        [0, 0, 2, 1, 3, 2, 2],
        [2, 2, 2, 1, 2, 2, 2],
        [2, 2, 2, 1, 

In [8]:
def train_model():
    X, y, student_codes = build_dataset()
    print("Số lượng samples:", len(X))
    print("Số features:", X.shape[1])
    print("Số labels:", y.shape[1])
    if len(X) < 2:
        raise ValueError("Cần ít nhất 2 samples để train/test split.")
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )
    base_model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    )
    model = MultiOutputClassifier(base_model)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print("\n=== Đánh giá từng năng lực ===")
    for i, field in enumerate(LABEL_FIELDS):
        print(f"\n--- {field} ---")
        print("Accuracy:", accuracy_score(y_test[:, i], y_pred[:, i]))
        print(
            classification_report(
                y_test[:, i],
                y_pred[:, i],
                labels=[0, 1, 2, 3, 4],
                target_names=[
                    "Không có",
                    "Chưa đủ cơ sở",
                    "Đang hình thành",
                    "Đạt",
                    "Nổi trội"
                ],
                zero_division=0
            )
        )

    joblib.dump(model, MODEL_PATH)
    print(f"\nĐã lưu model tại: {MODEL_PATH}")
    return model

In [9]:
def predict_one(model, json_path):
    features = load_input(json_path)
    prediction = model.predict([features])[0]
    result = []
    for field, level_id in zip(LABEL_FIELDS, prediction):
        result.append({
            "field": field,
            "level": REVERSE_LEVEL_MAPPING[int(level_id)]
        })
    return result

In [10]:
model = train_model()
test_file = "../json/20252026.01.Mo1.001.json"
if os.path.exists(test_file):
    result = predict_one(model, test_file)
    print("\n=== Kết quả dự đoán thử ===")
    print(json.dumps(result, ensure_ascii=False, indent=2))

Bỏ qua 20252026.03.Ba1.001: không tìm thấy label
Bỏ qua 20252026.03.Ba1.002: không tìm thấy label
Bỏ qua 20252026.03.Ba1.004: không tìm thấy label
Bỏ qua 20252026.03.Ba1.005: không tìm thấy label
Bỏ qua 20252026.03.Ba1.006: không tìm thấy label
Bỏ qua 20252026.03.Ba1.007: không tìm thấy label
Bỏ qua 20252026.03.Ba1.008: không tìm thấy label
Bỏ qua 20252026.03.Ba1.009: không tìm thấy label
Bỏ qua 20252026.03.Ba1.010: không tìm thấy label
Bỏ qua 20252026.03.Ba1.011: không tìm thấy label
Bỏ qua 20252026.03.Ba1.012: không tìm thấy label
Bỏ qua 20252026.03.Ba1.013: không tìm thấy label
Bỏ qua 20252026.03.Ba1.014: không tìm thấy label
Bỏ qua 20252026.03.Ba1.015: không tìm thấy label
Bỏ qua 20252026.03.Ba1.016: không tìm thấy label
Bỏ qua 20252026.03.Ba1.017: không tìm thấy label
Bỏ qua 20252026.03.Ba1.018: không tìm thấy label
Bỏ qua 20252026.03.Ba1.019: không tìm thấy label
Bỏ qua 20252026.03.Ba1.020: không tìm thấy label
Bỏ qua 20252026.03.Ba1.021: không tìm thấy label
Bỏ qua 20252026.03.B